# PG4 (gastric cancer, NCC): protein-adjusted phosphosite analysis

CDAP iTRAQ tables in which every column is already a log2 tumor/normal ratio for one patient
(80 patients, 50 of them measured twice). Tumor vs normal is therefore a one-sample test of whether
the mean log ratio differs from 0.

Technical replicates are averaged per patient first; treating them as separate patients would
inflate the sample size.

## Setup

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import phosphokit as pk

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
DATA = Path("../../data/raw/PG4")
PHOSPHO_FILE = DATA / "pg4_phosphoproteomics_data.xlsx"
PROTEIN_FILE = DATA / "pg4_proteomics_data.xlsx"
DATASET = "PG4"
OUT_DIR = Path(f"../../results/{DATASET}")
FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# These ratios are already relative to each patient's own normal sample, so normalization only has
# to remove per-sample loading differences.
S = pk.Settings(normalization="median_common", missing_method="none", min_per_group=10)
PROTEIN_VALUE = "Unshared Log "   # peptides unique to one protein


def savefig(name):
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{name}.png", bbox_inches="tight")
    plt.show()

## 1. Load and average technical replicates

In [ ]:
ph_ratio, ph_annot = pk.io.read_cdap_ratios(PHOSPHO_FILE, "Phosphosite", "Log ",
                                            annot_cols=["Gene", "Peptide"])
prot_ratio, _ = pk.io.read_cdap_ratios(PROTEIN_FILE, "Gene", PROTEIN_VALUE)
prot_ratio = prot_ratio[ph_ratio.columns]

samples = pk.io.cdap_sample_table(ph_ratio.columns)
print(f"{ph_ratio.shape[0]} sites, {prot_ratio.shape[0]} proteins, {ph_ratio.shape[1]} runs "
      f"from {samples['Case'].nunique()} patients")

ph_norm = pk.normalize(ph_ratio, S.normalization, S.norm_common_frac)
prot_norm = pk.normalize(prot_ratio, S.normalization, S.norm_common_frac)

# replicate agreement before averaging
pairs = samples.groupby("Case").filter(lambda g: len(g) == 2)
rep_r = pd.Series({case: ph_norm[g.index[0]].corr(ph_norm[g.index[1]]) for case, g in pairs.groupby("Case")})
print(f"median replicate correlation (phospho): {rep_r.median():.2f}")


def to_patient_level(mat):
    units, _ = pk.collapse_patients(mat, samples)
    units.columns = [c.split("|")[0] for c in units.columns]
    return units


ph_pat, prot_pat = to_patient_level(ph_norm), to_patient_level(prot_norm)
prot_pat = prot_pat[ph_pat.columns]
print("patient-level matrices:", ph_pat.shape, prot_pat.shape)

## 2. Protein adjustment input

In [ ]:
site_to_protein = ph_annot["Gene"]
ph_adj = pk.adjusted_matrix(ph_pat, prot_pat, site_to_protein)
protein_by_site = prot_pat.loc[site_to_protein[ph_adj.index].values].set_axis(ph_adj.index)

fig, ax = plt.subplots(figsize=(5.5, 3.6))
site_r = pk.phospho_protein_correlation(ph_pat.loc[ph_adj.index], protein_by_site, ax)
savefig("01_phospho_protein_correlation")

## 3. One-sample tests

An intercept-only model: the ratios are already paired within patient.

In [ ]:
res, info = {}, {}
res["protein"], info["protein"] = pk.one_sample_test(prot_pat, S)
res["phospho"], info["phospho"] = pk.one_sample_test(ph_pat, S)
res["adjA"], info["adjA"] = pk.one_sample_test(ph_adj, S)
res["adjB"] = pk.method_b(res["phospho"], res["protein"], site_to_protein)

overview = pd.DataFrame([{"analysis": k, "tested": len(r),
                          "Up": (pk.direction(r, S) == "Up").sum(),
                          "Down": (pk.direction(r, S) == "Down").sum(),
                          "prior_d0": info.get(k, {}).get("d0")} for k, r in res.items()])
overview

## 4. QC and figures

In [ ]:
label = ph_annot["Gene"].astype(str).to_dict()
fig, axes = plt.subplots(1, 4, figsize=(20, 4.8))
for ax, (k, title, lab) in zip(axes, [("protein", "Protein", None),
                                      ("phospho", "Phosphosite (unadjusted)", label),
                                      ("adjA", "Adjusted, Method A", label),
                                      ("adjB", "Adjusted, Method B", label)]):
    pk.volcano(res[k], ax, title, labels=lab, settings=S)
savefig("02_volcano")

pk.pvalue_histograms(res)
savefig("03_pvalue_histograms")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
pk.method_comparison(res["adjA"], res["adjB"], axes[0], axes[1], site_correlation=site_r)
savefig("04_methodA_vs_B")

## 5. Tables

In [ ]:
def pick(result, prefix, cols=("logFC", "P", "adj.P")):
    """Selected columns of a result table, prefixed, plus the Up/Down call."""
    out = result[list(cols)].add_prefix(f"{prefix}_")
    out[f"{prefix}_direction"] = pk.direction(result, S)
    return out


table = ph_annot.loc[res["adjA"].index].join(pick(res["phospho"], "Phospho"))
table = table.join(pick(res["protein"], "Protein", cols=("logFC", "P")), on="Gene")
table = table.join(pick(res["adjA"], "Adjusted"))
table["Adjusted_MethodB_direction"] = pk.direction(res["adjB"], S).reindex(table.index)
table["Category"] = pk.category_table(res["protein"], res["phospho"], res["adjA"], site_to_protein, S)["category"]
table = table.sort_values("Adjusted_P")

phospho_dep = ph_annot.join(pick(res["phospho"], "Phospho"), how="inner")
phospho_dep["Protein_quantified"] = phospho_dep.index.isin(ph_adj.index)
phospho_dep = phospho_dep[phospho_dep["Phospho_direction"] != ""].sort_values("Phospho_P")
adjusted_dep = table[table["Adjusted_direction"] != ""]
print(table["Category"].value_counts())
print(f"phospho DEP {len(phospho_dep)} | adjusted DEP {len(adjusted_dep)}")

out_file = OUT_DIR / f"{DATASET}_phospho_results.xlsx"
with pd.ExcelWriter(out_file) as xw:
    table.to_excel(xw, sheet_name="Adjusted_site_table")
    phospho_dep.to_excel(xw, sheet_name="Phospho_DEP")
    adjusted_dep.to_excel(xw, sheet_name="Adjusted_DEP")
    ph_adj.round(4).to_excel(xw, sheet_name="Adjusted_matrix_log2")
    pick(res["protein"], "Protein").sort_values("Protein_P").to_excel(xw, sheet_name="Protein")
    overview.to_excel(xw, sheet_name="Overview", index=False)
print("saved:", out_file)